# Benchmarking decoders in `bloqade-lanes`

This notebook shows a user-level workflow for comparing multiple decoders across multiple logical kernels.

The core idea is:

1. compile each kernel into a `GeminiLogicalSimulatorTask`
2. run the task with noise to get `detectors` and `observables`
3. run the same task without noise to get a reference logical output
4. evaluate each decoder on each kernel using a list of metric functions

The sketch in the design notes used a metric signature that only took `(decoder_ctor, error_model, detectors, observables)`. In practice, metrics such as logical error rate are easier to define if the metric also receives a small `context` dictionary containing the noiseless reference observables and a few run settings.


In [1]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from time import perf_counter
from typing import Any, Callable
import inspect
import sys
import tracemalloc

import numpy as np
import numpy.typing as npt
import stim

from bloqade import qubit, squin
from bloqade.gemini import logical
from bloqade.lanes import GeminiLogicalSimulator
from bloqade.decoders import (
    BaseDecoder,
    BeliefFindDecoder,
    BpLsdDecoder,
    BpOsdDecoder,
    MWPFDecoder,
)


In [2]:
DecoderConstructor = Callable[[stim.DetectorErrorModel], BaseDecoder]
MetricFn = Callable[
    [DecoderConstructor, stim.DetectorErrorModel, npt.NDArray[np.bool_], npt.NDArray[np.bool_], dict[str, Any]],
    list[float],
]


def _decoder_name(decoder_ctor: DecoderConstructor) -> str:
    return getattr(decoder_ctor, "__name__", decoder_ctor.__class__.__name__)


def _mode_row(obs: npt.NDArray[np.bool_]) -> npt.NDArray[np.bool_]:
    counts = Counter(map(lambda row: tuple(map(int, row)), obs.tolist()))
    row, _ = counts.most_common(1)[0]
    return np.asarray(row, dtype=bool)


def _decode_and_correct(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
) -> tuple[BaseDecoder, npt.NDArray[np.bool_], npt.NDArray[np.bool_]]:
    decoder = decoder_ctor(error_model)
    flips = np.asarray(decoder.decode(detectors), dtype=bool)
    corrected = np.asarray(observables, dtype=bool) ^ flips
    return decoder, flips, corrected


def benchmark_decoders(
    decoders: list[DecoderConstructor],
    kernels: list[Callable[..., Any]],
    metrics_fns: dict[str, MetricFn],
    *,
    shots: int = 10_000,
    reference_shots: int = 2_000,
    with_noise: bool = True,
) -> dict[str, dict[str, list[list[float]]]]:
    """
    Benchmark multiple decoders on multiple kernels.

    Returns:
        results[decoder_name][metric_name][kernel_index] = list[float]
    """
    simulator = GeminiLogicalSimulator()
    results: dict[str, dict[str, list[list[float]]]] = {
        _decoder_name(decoder): {metric_name: [] for metric_name in metrics_fns}
        for decoder in decoders
    }

    for kernel in kernels:
        task = simulator.task(kernel)
        noisy_result = task.run(shots, with_noise=with_noise)
        ref_result = task.run(reference_shots, with_noise=False)

        detectors = np.asarray(noisy_result.detectors, dtype=bool)
        observables = np.asarray(noisy_result.observables, dtype=bool)
        reference_observables = np.asarray(ref_result.observables, dtype=bool)

        context = {
            "kernel_name": getattr(kernel, "sym_name", getattr(kernel, "__name__", "kernel")),
            "shots": shots,
            "reference_shots": reference_shots,
            "reference_observables": reference_observables,
            "target_observables": _mode_row(reference_observables),
        }

        for decoder_ctor in decoders:
            decoder_name = _decoder_name(decoder_ctor)
            for metric_name, metric_fn in metrics_fns.items():
                metric_values = metric_fn(
                    decoder_ctor,
                    task.detector_error_model,
                    detectors,
                    observables,
                    context,
                )
                results[decoder_name][metric_name].append(metric_values)

    return results


In [3]:
def logical_error_rate_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    _, _, corrected = _decode_and_correct(decoder_ctor, error_model, detectors, observables)
    target = np.broadcast_to(context["target_observables"], corrected.shape)
    failures = np.any(corrected != target, axis=1)
    return [float(np.mean(failures))]


def latency_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    t0 = perf_counter()
    decoder = decoder_ctor(error_model)
    t1 = perf_counter()
    _ = decoder.decode(detectors)
    t2 = perf_counter()
    return [1e3 * (t1 - t0), 1e3 * (t2 - t1)]


def throughput_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    decoder = decoder_ctor(error_model)
    t0 = perf_counter()
    _ = decoder.decode(detectors)
    dt = perf_counter() - t0
    shots_per_second = float(len(detectors) / dt) if dt > 0 else float("inf")
    return [shots_per_second]


def robustness_to_model_mismatch_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    mismatch_prob = context.get("mismatch_prob", 0.02)
    target = np.broadcast_to(context["target_observables"], observables.shape)

    decoder = decoder_ctor(error_model)
    clean_flips = np.asarray(decoder.decode(detectors), dtype=bool)
    clean_corrected = observables ^ clean_flips
    clean_ler = float(np.mean(np.any(clean_corrected != target, axis=1)))

    rng = np.random.default_rng(0)
    detector_noise = rng.random(detectors.shape) < mismatch_prob
    mismatched_detectors = detectors ^ detector_noise
    mismatch_flips = np.asarray(decoder.decode(mismatched_detectors), dtype=bool)
    mismatch_corrected = observables ^ mismatch_flips
    mismatch_ler = float(np.mean(np.any(mismatch_corrected != target, axis=1)))
    return [clean_ler, mismatch_ler, mismatch_ler - clean_ler]


def memory_hardware_cost_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    tracemalloc.start()
    decoder = decoder_ctor(error_model)
    _ = decoder.decode(detectors)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    shallow_size = float(sys.getsizeof(decoder))
    return [shallow_size / (1024**2), peak / (1024**2)]


INTERPRETABILITY_PRIORS = {
    "BeliefFindDecoder": (0.65, 0.55),
    "BpLsdDecoder": (0.55, 0.60),
    "BpOsdDecoder": (0.45, 0.72),
    "MWPFDecoder": (0.35, 0.78),
    "TesseractDecoder": (0.30, 0.85),
}


def interpretability_complexity_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    name = _decoder_name(decoder_ctor)
    if name in INTERPRETABILITY_PRIORS:
        interpretability, complexity = INTERPRETABILITY_PRIORS[name]
    else:
        n_params = len(inspect.signature(decoder_ctor).parameters)
        complexity = min(1.0, 0.4 + 0.05 * n_params)
        interpretability = max(0.0, 1.0 - complexity)
    return [interpretability, complexity]


metrics = {
    "logical_error_rate": logical_error_rate_metric,
    "latency": latency_metric,
    "throughput": throughput_metric,
    "robustness_to_model_mismatch": robustness_to_model_mismatch_metric,
    "memory_hardware_cost": memory_hardware_cost_metric,
    "interpretability_implementation_complexity": interpretability_complexity_metric,
}


## Example kernels

These are intentionally small kernels so the benchmarking loop is easy to run and modify.


In [4]:
@logical.kernel(aggressive_unroll=True, verify=True)
def ghz3_kernel():
    reg = qubit.qalloc(3)
    squin.h(reg[0])
    squin.cx(reg[0], reg[1])
    squin.cx(reg[1], reg[2])
    logical.default_post_processing(reg)


@logical.kernel(aggressive_unroll=True, verify=True)
def bell2_kernel():
    reg = qubit.qalloc(2)
    squin.h(reg[0])
    squin.cx(reg[0], reg[1])
    logical.default_post_processing(reg)


In [5]:
decoders = [BpLsdDecoder]
kernels = [ghz3_kernel]

benchmark_results = benchmark_decoders(
    decoders=decoders,
    kernels=kernels,
    metrics_fns=metrics,
    shots=2_000,
    reference_shots=512,
)

benchmark_results


{'BpLsdDecoder': {'logical_error_rate': [[0.5175]],
  'latency': [[0.4995420022169128, 31.277500005671754]],
  'throughput': [[64521.2449529475]],
  'robustness_to_model_mismatch': [[0.5175, 0.57, 0.05249999999999999]],
  'memory_hardware_cost': [[4.57763671875e-05, 0.5879220962524414]],
  'interpretability_implementation_complexity': [[0.55, 0.6]]}}

In [6]:
def print_metric_table(results: dict[str, dict[str, list[list[float]]]], metric_name: str, kernels: list[Callable[..., Any]]):
    kernel_names = [getattr(kernel, "sym_name", getattr(kernel, "__name__", "kernel")) for kernel in kernels]
    print(metric_name)
    print("=" * len(metric_name))
    for decoder_name, decoder_results in results.items():
        print(f"\\n{decoder_name}:")
        for kernel_name, values in zip(kernel_names, decoder_results[metric_name]):
            print(f"  {kernel_name}: {values}")


print_metric_table(benchmark_results, "logical_error_rate", kernels)
print_metric_table(benchmark_results, "throughput", kernels)


logical_error_rate
\nBpLsdDecoder:
  ghz3_kernel: [0.5175]
throughput
\nBpLsdDecoder:
  ghz3_kernel: [64521.2449529475]


## Example: custom metric functions

Users can add their own metrics as long as they follow the `MetricFn` signature used above.


In [7]:
def average_flips_metric(
    decoder_ctor: DecoderConstructor,
    error_model: stim.DetectorErrorModel,
    detectors: npt.NDArray[np.bool_],
    observables: npt.NDArray[np.bool_],
    context: dict[str, Any],
) -> list[float]:
    _, flips, _ = _decode_and_correct(decoder_ctor, error_model, detectors, observables)
    return [float(np.mean(flips))]


custom_metrics = {
    "logical_error_rate": logical_error_rate_metric,
    "average_flips": average_flips_metric,
}

custom_results = benchmark_decoders(
    decoders=[BpLsdDecoder, BpOsdDecoder],
    kernels=[ghz3_kernel],
    metrics_fns=custom_metrics,
    shots=1_000,
    reference_shots=256,
)

custom_results


{'BpLsdDecoder': {'logical_error_rate': [[0.508]],
  'average_flips': [[0.06633333333333333]]},
 'BpOsdDecoder': {'logical_error_rate': [[0.508]],
  'average_flips': [[0.06633333333333333]]}}